# tool_02 提交调度

## 功能
1. **生成提交文件**：job_list.txt、各任务 submit 脚本
2. **核数配给**：按分区方案分配每任务核数
3. **Hub 分区方案**：young.ng（Slurm）、young（SEG）等，灵活选择
4. **快速排任务**：支持单任务/批量/分片提交

In [ ]:
import os
from pathlib import Path

In [ ]:
# ============================================
# 从 shared/config.py 读取（clio 可直接修改 config.py）
# ============================================
import sys
from pathlib import Path
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "shared" / "config.py").exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
from shared.load_config import load_config
cfg = load_config()
zpe_root = cfg.ZPE_ROOT
DEFAULT_PARTITION = getattr(cfg, 'DEFAULT_PARTITION', 'young')

In [ ]:
# ============================================
# Hub 分区方案（GR_main 资源模板）
# ============================================
HUB_PARTITIONS = {
    "young_ng": {
        "scheduler": "slurm",
        "nodes": 3,
        "tasks_per_node": 40,
        "total_cores": 120,  # 40 的整数倍
        "module": "module load vasp/5.4.4-18apr2017-vtst-r178/intel-2017-update1",
        "mpi_cmd": "srun vasp_std",
    },
    "young": {
        "scheduler": "sge",
        "total_cores": 128,
        "pe": "mpi",
        "module": "module load vasp/5.4.4-18apr2017-vtst-r178/intel-2017-update1",
        "mpi_cmd": "gerun vasp_std",
    },
}

# DEFAULT_PARTITION 已从 config.py 加载

In [ ]:
def scan_job_dirs(zpe_root):
    """扫描含 POSCAR+INCAR 的任务目录。"""
    job_dirs = []
    zpe = Path(zpe_root)
    if not zpe.exists():
        return []
    for root, dirs, _ in os.walk(zpe):
        r = Path(root)
        if (r / "POSCAR").exists() and (r / "INCAR").exists():
            job_dirs.append(root)
    return sorted(set(job_dirs))


def gen_sge_submit(job_dir, job_name, partition="young"):
    cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young"])
    cores = cfg["total_cores"]
    pe = cfg.get("pe", "mpi")
    return f"""#!/bin/bash
#$ -N {job_name}
#$ -S /bin/bash
#$ -l h_rt=2:00:00
#$ -P Gold
#$ -A UCL_chemM_Catlow
#$ -pe {pe} {cores}
#$ -cwd

source /etc/profile.d/modules.sh
{cfg['module']}

cd {job_dir}
{cfg['mpi_cmd']} > result.$JOB_ID
"""


def gen_slurm_submit(job_dir, job_name, partition="young_ng"):
    cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young_ng"])
    nodes = cfg.get("nodes", 3)
    tpn = cfg.get("tasks_per_node", 40)
    return f"""#!/bin/bash
#SBATCH -J {job_name}
#SBATCH -N {nodes}
#SBATCH --ntasks-per-node={tpn}
#SBATCH -t 02:00:00
#SBATCH -p <partition_name>

source /etc/profile.d/modules.sh
{cfg['module']}

cd {job_dir}
{cfg['mpi_cmd']} > result.$SLURM_JOB_ID
"""


def gen_array_submit_sge(job_list_path, job_dirs, partition="young"):
    cfg = HUB_PARTITIONS.get(partition, HUB_PARTITIONS["young"])
    cores = cfg["total_cores"]
    pe = cfg.get("pe", "mpi")
    n = len(job_dirs)
    return f"""#!/bin/bash
#$ -N ZPE_frequency
#$ -S /bin/bash
#$ -l h_rt=2:00:00
#$ -P Gold
#$ -A UCL_chemM_Catlow
#$ -pe {pe} {cores}
#$ -t 1-{n}
#$ -cwd

source /etc/profile.d/modules.sh
{cfg['module']}

JOB_LIST="{job_list_path}"
JOB_DIR=$(sed -n "${{SGE_TASK_ID}}p" "$JOB_LIST")
JOB_NAME=$(basename "$JOB_DIR")
cd "$JOB_DIR"
{cfg['mpi_cmd']} > result.$JOB_ID.$SGE_TASK_ID.$JOB_NAME
"""


In [ ]:
# 提交前检查（原子固定、INCAR）— 不通过则禁止提交
from shared.pre_check import run_pre_check
if not run_pre_check(zpe_root):
    raise RuntimeError("提交前检查未通过，请先修正后再提交。详见 频率计算/shared/提交前检查.md")

In [ ]:
# 扫描任务目录
job_dirs = scan_job_dirs(zpe_root)
print(f"📌 扫描到 {len(job_dirs)} 个任务目录")
for d in job_dirs:
    print(f"  - {os.path.relpath(d, zpe_root)}")

In [ ]:
# 生成 job_list.txt
job_list_path = os.path.join(zpe_root, "job_list.txt")
with open(job_list_path, "w") as f:
    for d in job_dirs:
        f.write(os.path.abspath(d) + "\n")
print(f"✔ job_list.txt: {job_list_path}")

# 生成批量提交脚本（SGE 数组任务）
partition = DEFAULT_PARTITION
cfg = HUB_PARTITIONS[partition]
sched = cfg["scheduler"]

if sched == "sge":
    submit_content = gen_array_submit_sge(os.path.abspath(job_list_path), job_dirs, partition)
    submit_path = os.path.join(zpe_root, "submit.sh")
    with open(submit_path, "w") as f:
        f.write(submit_content)
    print(f"✔ submit.sh (SGE 数组): {submit_path}")
    print(f"  分区: {partition}, 每任务 {cfg['total_cores']} 核")
else:
    # Slurm: 可为每个任务生成单独脚本，或后续扩展数组
    submit_path = os.path.join(zpe_root, "submit.sh")
    with open(submit_path, "w") as f:
        f.write("# Slurm 批量提交：请根据本机配置填写 #SBATCH -p 分区名\n")
        f.write(f"# 任务数: {len(job_dirs)}")
    print(f"✔ submit.sh 占位已生成，Slurm 需按本机分区修改")

print(f"\n>>> 提交命令: cd {zpe_root} && qsub submit.sh")

In [ ]:
# 可选：为每个任务生成单独 submit 脚本（便于分片、多机调度）
os.makedirs(os.path.join(zpe_root, "submit_scripts"), exist_ok=True)
for i, jd in enumerate(job_dirs):
    jn = os.path.basename(jd)
    cfg = HUB_PARTITIONS[DEFAULT_PARTITION]
    if cfg["scheduler"] == "sge":
        content = gen_sge_submit(jd, jn, DEFAULT_PARTITION)
    else:
        content = gen_slurm_submit(jd, jn, DEFAULT_PARTITION)
    sp = os.path.join(zpe_root, "submit_scripts", f"submit_{jn}.sh")
    with open(sp, "w") as f:
        f.write(content)
print(f"✔ 单独脚本: submit_scripts/submit_*.sh ({len(job_dirs)} 个)")